#### imorts

In [17]:
import joblib
import pandas as pd

#### load models

In [18]:
diagnosis_model = joblib.load(
    "../models/malaria_diagnosis_rf.pkl"
)

severity_model = joblib.load(
    "../models/malaria_severity_rf.pkl"
)

#### verify 

In [19]:
print(type(diagnosis_model))
print(type(severity_model))

<class 'sklearn.pipeline.Pipeline'>
<class 'sklearn.pipeline.Pipeline'>


## create diagnosis prediction function

In [21]:
def predict_diagnosis(patient_data):

    patient_df = pd.DataFrame([patient_data])

    prediction = diagnosis_model.predict(patient_df)[0]

    probability = diagnosis_model.predict_proba(
        patient_df
    )[0]

    return {
        "diagnosis": prediction,
        "probability": max(probability)
    }

## create severity prediction function

In [22]:
def predict_severity(patient_data):

    patient_df = pd.DataFrame([patient_data])

    prediction = severity_model.predict(
        patient_df
    )[0]

    probability = severity_model.predict_proba(
        patient_df
    )[0]

    return {
        "severity": prediction,
        "probability": max(probability)
    }

### create label mapping
this was our earlier encoding 

In [23]:
{'Moderate': 0,
 'Severe': 1,
 'Uncomplicated': 2}

{'Moderate': 0, 'Severe': 1, 'Uncomplicated': 2}

and this is our diagnosis

Malaria = 0
No Malaria = 1

#### now lets convert prediction to readable text

In [24]:
diagnosis_map = {
    0: "Malaria",
    1: "No Malaria"
}

severity_map = {
    0: "Moderate",
    1: "Severe",
    2: "Uncomplicated"
}

## triage function

In [25]:
def malaria_triage(patient_data):

    diagnosis_result = predict_diagnosis(
        patient_data
    )

    diagnosis_label = diagnosis_map[
        diagnosis_result["diagnosis"]
    ]

    if diagnosis_label == "No Malaria":

        return {
            "Diagnosis": "No Malaria",
            "Confidence":
                round(
                    diagnosis_result["probability"] * 100,
                    2
                ),
            "Severity": "N/A",
            "Recommendation":
                "Monitor symptoms and consult a healthcare provider if symptoms persist."
        }

    severity_result = predict_severity(
        patient_data
    )

    severity_label = severity_map[
        severity_result["severity"]
    ]

    if severity_label == "Severe":

        recommendation = (
            "Immediate hospital admission required."
        )

    elif severity_label == "Moderate":

        recommendation = (
            "Seek medical attention within 24 hours."
        )

    else:

        recommendation = (
            "Outpatient treatment and monitoring recommended."
        )

    return {
        "Diagnosis": diagnosis_label,
        "Confidence":
            round(
                diagnosis_result["probability"] * 100,
                2
            ),
        "Severity": severity_label,
        "Recommendation": recommendation
    }

### create sample patient

In [26]:
sample_patient = {
    "age": 15,
    "sex": "F",
    "pregnant": False,
    "location": "Sub-Saharan Africa - Rural",
    "travel_history_endemic_area": True,
    "bed_net_use": False,
    "irs_spraying": False,
    "previous_malaria_episodes": 3,
    "fever": True,
    "chills_rigors": True,
    "headache": True,
    "night_sweats": True,
    "fatigue_malaise": True,
    "nausea_vomiting": True,
    "diarrhea": False,
    "cough": False,
    "abdominal_pain": True,
    "jaundice": False,
    "altered_consciousness": False,
    "seizures": False,
    "hemoglobin_g_dl": 10.6,
    "platelets_x10e9_l": 132,
    "wbc_x10e9_l": 5.0,
    "glucose_mg_dl": 106,
    "creatinine_mg_dl": 0.34,
    "bilirubin_mg_dl": 1.5,
    "lactate_mmol_l": 1.8
}

#### test

In [27]:
result = malaria_triage(
    sample_patient
)

print(result)

{'Diagnosis': 'Malaria', 'Confidence': np.float64(99.47), 'Severity': 'Uncomplicated', 'Recommendation': 'Outpatient treatment and monitoring recommended.'}
